# Import

In [1]:
import numpy as np

# Rank via Elimination หาลำดับขั้นในการ elimination

In [2]:
def rank_matrix(M):
    A = M.astype(float).copy()
    rows, cols = A.shape
    rank = 0

    for col in range(cols):
        pivot_row = -1
        for r in range(rank, rows):
            if abs(A[r, col]) > 1e-10:
                pivot_row = r
                break

        if pivot_row != -1:
            A[[rank, pivot_row]] = A[[pivot_row, rank]]
            A[rank] = A[rank] / A[rank, col]

            for r in range(rows):
                if r != rank:
                    A[r] -= A[r, col] * A[rank]

            rank += 1

    return rank


# Consistency Check

In [3]:
def check_solution_type(A, b):
    Ab = np.hstack([A, b.reshape(-1, 1)])
    rankA = rank_matrix(A)
    rankAb = rank_matrix(Ab)

    if rankA < rankAb:
        return "No"
    elif rankA < A.shape[1]:
        return "Infinite"
    else:
        return "Unique"


# Solution Reporter

In [4]:
def print_result(method, x, status):
    print(f"\n[{method}]")
    if status == "Unique" and x is not None:
        print("✅ ระบบสมการนี้มีคำตอบเดียว")
        print("x =")
        print(x)
    elif status == "Infinite":
        print("⚠️ ระบบสมการนี้มีคำตอบไม่สิ้นสุด (Infinite Solutions)")
    elif status == "No":
        print("❌ ระบบสมการนี้ไม่มีคำตอบ (No Solution)")
    else:
        print("❌ ไม่สามารถหาคำตอบได้")


# Gaussian Elimination

In [5]:
def gauss_elimination_pivot(A, b):
    n = len(b)
    Ab = np.hstack([A.astype(float), b.reshape(-1, 1)])

    for i in range(n):
        max_row = i
        for r in range(i+1, n):
            if abs(Ab[r, i]) > abs(Ab[max_row, i]):
                max_row = r
        Ab[[i, max_row]] = Ab[[max_row, i]]

        if abs(Ab[i, i]) < 1e-10:
            return None

        for r in range(i+1, n):
            factor = Ab[r, i] / Ab[i, i]
            Ab[r] -= factor * Ab[i]

    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (Ab[i, -1] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]

    return x


# Gauss Jordan

In [6]:
def gauss_jordan(A, b):
    n = len(b)
    Ab = np.hstack([A.astype(float), b.reshape(-1, 1)])

    for i in range(n):
        if abs(Ab[i, i]) < 1e-10:
            return None

        Ab[i] = Ab[i] / Ab[i, i]

        for r in range(n):
            if r != i:
                Ab[r] -= Ab[r, i] * Ab[i]

    return Ab[:, -1]


# LU Factorization

In [7]:
def lu_factorization(A, b):
    n = A.shape[0]
    L = np.eye(n)
    U = A.astype(float).copy()

    for i in range(n):
        if abs(U[i, i]) < 1e-10:
            return None

        for r in range(i+1, n):
            factor = U[r, i] / U[i, i]
            L[r, i] = factor
            U[r] -= factor * U[i]

    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - sum(L[i][j] * y[j] for j in range(i))

    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (y[i] - sum(U[i][j] * x[j] for j in range(i+1, n))) / U[i][i]

    return x



# Multiply Vector

In [8]:
def multiply_matrix_vector(A, b):
    n = len(b)
    x = np.zeros(n)
    for i in range(n):
        for j in range(n):
            x[i] += A[i][j] * b[j]
    return x

# Inverse Matrix

In [9]:
def inverse_matrix(A):
    n = A.shape[0]
    AI = np.hstack([A.astype(float), np.eye(n)])

    for i in range(n):
        if abs(AI[i, i]) < 1e-10:
            return None

        AI[i] = AI[i] / AI[i, i]

        for r in range(n):
            if r != i:
                AI[r] -= AI[r, i] * AI[i]

    return AI[:, n:]


# INPUT FUNCTIONS

In [10]:
# ---------------- INPUT FUNCTIONS ---------------- #
def input_matrix():
    while True:
        try:
            n = int(input("\nกรอกขนาดของ Matrix A (n x n): "))
            if n <= 0:
                print("⚠️ n ต้องมากกว่า 0")
                continue

            A = []
            print("กรอกค่า Matrix A ทีละแถว")

            for i in range(n):
                while True:
                    row = list(map(float, input(f"แถวที่ {i+1}: ").split()))

                    if len(row) != n:
                        print(f"⚠️ ต้องกรอก {n} ตัว")
                    else:
                        A.append(row)
                        break

            return np.array(A)

        except ValueError:
            print("⚠️ กรุณากรอกตัวเลขเท่านั้น")


def input_vector(n):
    while True:
        try:
            b = list(map(float, input("กรอก vector b: ").split()))

            if len(b) != n:
                print(f"⚠️ vector b ต้องมี {n} ตัว")
            else:
                return np.array(b)

        except ValueError:
            print("⚠️ กรุณากรอกตัวเลขเท่านั้น")


# ---------------- MAIN PROGRAM ---------------- #

while True:
    print("\n==============================")
    print("โปรแกรมแก้ระบบสมการเชิงเส้น")
    print("1. กรอกสมการใหม่")
    print("0. จบการทำงาน")
    print("==============================")

    choice = input("เลือกเมนู: ")

    if choice == '0':
        print("จบการทำงานของโปรแกรม")
        break

    elif choice == '1':

        A = input_matrix()
        n = len(A)

        # 🔥 เช็ค singular เร็ว ๆ
        if abs(np.linalg.det(A)) < 1e-12:
            print("\n⚠️ Matrix นี้อาจเป็น Singular (det ≈ 0)")
            print("บาง method อาจไม่สามารถคำนวณได้")

        has_b = input("Matrix นี้มี vector b หรือไม่ (y/n): ").lower()

        if has_b == 'y':

            b = input_vector(n)

            status = check_solution_type(A, b)

            print_result("Gauss Elimination with Pivoting",
                         gauss_elimination_pivot(A, b), status)

            print_result("Gauss Jordan Elimination",
                         gauss_jordan(A, b), status)

            print_result("LU Factorization",
                         lu_factorization(A, b), status)

            print("\n[Inverse Matrix และ x จาก A⁻¹b]")

            invA = inverse_matrix(A)

            if invA is None:
                print("❌ ไม่สามารถหา Inverse Matrix ได้ (Matrix เป็น Singular)")
                print("❌ ไม่สามารถหาค่า x จาก A⁻¹b ได้")
            else:
                print("✅ Inverse Matrix =")
                print(invA)

                x_inv = multiply_matrix_vector(invA, b)
                print("✅ x = A⁻¹b =")
                print(x_inv)

        elif has_b == 'n':

            print("\n[Inverse Matrix]")
            invA = inverse_matrix(A)

            if invA is None:
                print("❌ ไม่สามารถหา Inverse Matrix ได้ (Matrix เป็น Singular)")
            else:
                print("✅ Inverse Matrix =")
                print(invA)

        else:
            print("⚠️ กรุณาเลือก y หรือ n เท่านั้น")

    else:
        print("⚠️ เลือกเมนูไม่ถูกต้อง กรุณาเลือกใหม่")



โปรแกรมแก้ระบบสมการเชิงเส้น
1. กรอกสมการใหม่
0. จบการทำงาน
จบการทำงานของโปรแกรม
